In [1]:
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings

In [2]:
warnings.filterwarnings('ignore')

In [3]:
AUDIO_PATH = '../data/ESC-50-master/ESC-50-master/audio/'
META_PATH = '../data/ESC-50-master/ESC-50-master/meta/esc50.csv'

df = pd.read_csv(META_PATH)
print(df.shape)
print(df.head())

(2000, 7)
            filename  fold  target        category  esc10  src_file take
0   1-100032-A-0.wav     1       0             dog   True    100032    A
1  1-100038-A-14.wav     1      14  chirping_birds  False    100038    A
2  1-100210-A-36.wav     1      36  vacuum_cleaner  False    100210    A
3  1-100210-B-36.wav     1      36  vacuum_cleaner  False    100210    B
4  1-101296-A-19.wav     1      19    thunderstorm  False    101296    A


In [4]:
def extract_mfcc(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfccs, axis=1)

features = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    file_path = os.path.join(AUDIO_PATH, row['filename'])
    mfcc = extract_mfcc(file_path)
    features.append(mfcc)
    labels.append(row['category'])

X = np.array(features)
y = np.array(labels)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

100%|██████████| 2000/2000 [00:48<00:00, 41.48it/s]

X shape: (2000, 13)
y shape: (2000,)


In [5]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')
print(f'Classes: {len(le.classes_)}')

Train size: (1600, 13)
Test size: (400, 13)
Classes: 50
